[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/24.%20The%20Static%20Truck%20Appointment%20System%20Problem/P24-Tier-5_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 24. The Static Truck Appointment System Problem

## Tier 5: The Integrated Digital Twin (System-of-Systems Simulation)

### Goal
Implement a comprehensive Digital Twin system that transforms the Static Truck Appointment System into a real-time simulation environment mirroring actual terminal operations. This system-of-systems approach integrates multiple subsystems: gate operations, yard management, equipment allocation, traffic flow, and external logistics networks.

### Key assumptions
- Real-time data synchronization with physical terminal operations
- IoT sensors, GPS tracking, and RFID systems provide continuous data streams
- Multiple time horizons: real-time monitoring, operational adjustments, tactical planning
- Predictive models for demand forecasting and congestion analysis
- What-if scenario analysis capabilities for proactive decision making

### Approach (step-by-step)
1. **Design system architecture** with multiple integrated components
2. **Implement real-time data ingestion** from simulated sensor networks
3. **Create dynamic simulation models** for gate processing and yard operations
4. **Build predictive analytics** for demand forecasting and congestion prediction
5. **Develop optimization engines** for dynamic appointment rescheduling
6. **Implement stakeholder dashboards** for operational visibility
7. **Demonstrate what-if scenarios** with measurable improvements

### What to look for in the results
- Real-time system monitoring with live KPIs and metrics
- Predictive analytics showing forecasted demand and congestion
- What-if scenario analysis demonstrating improvement potential
- Multi-system integration showing coordination between components
- Digital-physical synchronization with bi-directional data flow

### Concrete example (from the source)
The Digital Twin continuously synchronizes with physical terminal operations through IoT sensors, GPS tracking, RFID systems, and operational databases. This bi-directional data flow enables real-time system state monitoring, predictive analytics, and what-if scenario analysis for appointment scheduling decisions.

**Expected Scenario:** Peak morning operations at 9:00 AM with 15 incoming appointment requests and current system utilization at 85%.

**Expected Results:**
- Redistributed 6 appointments to less congested time slots
- Suggested proactive equipment reallocation to compensate for gate issues
- Achieved 23% reduction in predicted waiting times
- Maintained 96% customer satisfaction through proactive communication

In [1]:
from typing import Tuple, List, Dict, Set

# Import required libraries for Digital Twin implementation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from typing import List, Dict, Tuple, Optional, Callable
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from collections import deque
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully for Digital Twin implementation")

Libraries imported successfully for Digital Twin implementation


In [ ]:
# Define comprehensive data structures for Digital Twin
@dataclass
class TruckRequest:
    """Enhanced truck request with real-time tracking"""
    id: int
    preferred_time: int
    weight: float
    service_type: str = 'standard'
    carrier_id: int = 1
    cargo_type: str = 'standard'
    processing_time: float = 1.0
    gps_location: Tuple[float, float] = (0.0, 0.0)  # Lat, Lon
    estimated_arrival: datetime = field(default_factory=datetime.now)
    status: str = 'pending'  # pending, assigned, in_transit, at_gate, processing, completed

@dataclass
class GateSensor:
    """IoT sensor data from processing gates"""
    gate_id: int
    status: str = 'operational'  # operational, maintenance, malfunction
    current_truck: Optional[int] = None  # Truck ID being processed
    processing_time_remaining: float = 0.0
    queue_length: int = 0
    average_processing_time: float = 1.0
    utilization_24h: float = 0.0
    temperature: float = 25.0  # Celsius
    vibration_index: float = 0.1

@dataclass
class YardSystem:
    """Yard management system state"""
    total_containers: int = 1000
    available_spaces: int = 200
    yard_utilization: float = 0.8
    equipment_availability: Dict[str, int] = field(default_factory=lambda: {
        'yard_trucks': 8, 'cranes': 4, 'handlers': 12
    })
    congestion_level: float = 0.5
    weather_impact: float = 1.0

@dataclass
class TrafficSystem:
    """Traffic flow and external conditions"""
    current_congestion: float = 0.3  # 0-1 scale
    average_speed: float = 25.0  # km/h
    incidents: List[Dict] = field(default_factory=list)
    weather_condition: str = 'clear'
    visibility: float = 1.0  # 0-1 scale
    road_capacity_utilization: float = 0.6

@dataclass
class SystemKPIs:
    """Key Performance Indicators"""
    total_trucks_processed: int = 0
    average_waiting_time: float = 0.0
    gate_utilization: float = 0.0
    customer_satisfaction: float = 0.95
    on_time_performance: float = 0.9
    system_efficiency: float = 0.85
    carbon_emissions: float = 100.0  # kg CO2

print("Digital Twin data structures defined")

Digital Twin data structures defined


In [ ]:
# Real-time Data Simulation System
class RealTimeDataSimulator:
    """Simulates real-time data streams from IoT sensors and systems"""
    
    def __init__(self, num_gates: int = 8):
        self.num_gates = num_gates
        self.current_time = datetime.now()
        self.time_step = timedelta(minutes=5)  # 5-minute intervals
        
        # Initialize system components
        self.gate_sensors = [GateSensor(i) for i in range(num_gates)]
        self.yard_system = YardSystem()
        self.traffic_system = TrafficSystem()
        self.kpis = SystemKPIs()
        
        # Data streams
        self.truck_requests = deque()
        self.processing_history = deque(maxlen=1000)
        self.sensor_readings = deque(maxlen=500)
        
        # Random seed for reproducibility
        np.random.seed(42)
    
    def generate_truck_requests(self, num_requests: int = 5) -> List[TruckRequest]:
        """Generate new truck requests with realistic patterns"""
        requests = []
        
        for i in range(num_requests):
            # Time-based patterns (more requests during peak hours)
            hour = self.current_time.hour
            if 8 <= hour <= 10 or 14 <= hour <= 16:  # Peak hours
                request_probability = 0.8
            else:
                request_probability = 0.3
            
            if np.random.random() < request_probability:
                request_id = len(self.truck_requests) + i + 1
                preferred_time = np.random.randint(0, 8)
                weight = np.random.uniform(0.5, 2.5)
                service_type = np.random.choice(['standard', 'priority', 'overnight'], 
                                             p=[0.7, 0.2, 0.1])
                carrier_id = np.random.randint(1, 6)
                cargo_type = np.random.choice(['standard', 'refrigerated', 'hazardous'], 
                                             p=[0.6, 0.3, 0.1])
                
                # Simulated GPS location (around terminal)
                lat = 34.0522 + np.random.uniform(-0.1, 0.1)
                lon = -118.2437 + np.random.uniform(-0.1, 0.1)
                
                # Estimated arrival based on current traffic
                travel_time = np.random.uniform(15, 45)  # minutes
                eta = self.current_time + timedelta(minutes=travel_time)
                
                request = TruckRequest(
                    id=request_id,
                    preferred_time=preferred_time,
                    weight=weight,
                    service_type=service_type,
                    carrier_id=carrier_id,
                    cargo_type=cargo_type,
                    gps_location=(lat, lon),
                    estimated_arrival=eta
                )
                
                requests.append(request)
        
        return requests
    
    def update_gate_sensors(self):
        """Update gate sensor readings with realistic variations"""
        for sensor in self.gate_sensors:
            # Random gate status changes
            if np.random.random() < 0.02:  # 2% chance of malfunction
                sensor.status = 'malfunction'
            elif sensor.status == 'malfunction' and np.random.random() < 0.1:
                sensor.status = 'operational'
            
            # Update queue length and processing time
            if sensor.status == 'operational':
                sensor.queue_length = max(0, sensor.queue_length + np.random.randint(-1, 3))
                sensor.processing_time_remaining = max(0, 
                    sensor.processing_time_remaining - np.random.uniform(0, 0.5))
                
                # Environmental factors
                sensor.temperature += np.random.uniform(-2, 2)
                sensor.vibration_index = max(0, min(1, sensor.vibration_index + np.random.uniform(-0.1, 0.1)))
            
            # Calculate utilization
            sensor.utilization_24h = np.random.uniform(0.6, 0.95)
    
    def update_yard_system(self):
        """Update yard system state"""
        # Simulate container movements
        container_change = np.random.randint(-5, 10)
        self.yard_system.total_containers = max(800, min(1200, 
            self.yard_system.total_containers + container_change))
        
        self.yard_system.available_spaces = max(100, 
            200 - (self.yard_system.total_containers - 1000))
        self.yard_system.yard_utilization = self.yard_system.total_containers / 1200
        
        # Equipment availability changes
        for equipment in self.yard_system.equipment_availability:
            if np.random.random() < 0.05:  # 5% chance of equipment going down
                self.yard_system.equipment_availability[equipment] = max(0,
                    self.yard_system.equipment_availability[equipment] - 1)
            elif np.random.random() < 0.1:  # 10% chance of equipment coming back online
                max_equipment = {'yard_trucks': 10, 'cranes': 6, 'handlers': 15}[equipment]
                self.yard_system.equipment_availability[equipment] = min(max_equipment,
                    self.yard_system.equipment_availability[equipment] + 1)
        
        # Congestion and weather impacts
        self.yard_system.congestion_level = max(0, min(1, 
            self.yard_system.congestion_level + np.random.uniform(-0.1, 0.1)))
        
        weather_conditions = ['clear', 'rain', 'fog', 'wind']
        if np.random.random() < 0.1:  # 10% chance of weather change
            self.traffic_system.weather_condition = np.random.choice(weather_conditions)
        
        weather_impacts = {'clear': 1.0, 'rain': 0.8, 'fog': 0.6, 'wind': 0.9}
        self.yard_system.weather_impact = weather_impacts[self.traffic_system.weather_condition]
    
    def update_traffic_system(self):
        """Update traffic flow conditions"""
        # Time-based traffic patterns
        hour = self.current_time.hour
        if 7 <= hour <= 9 or 16 <= hour <= 18:  # Rush hours
            target_congestion = np.random.uniform(0.7, 0.9)
        elif 10 <= hour <= 15:  # Off-peak
            target_congestion = np.random.uniform(0.2, 0.4)
        else:  # Evening
            target_congestion = np.random.uniform(0.4, 0.6)
        
        # Gradual congestion changes
        self.traffic_system.current_congestion += (target_congestion - self.traffic_system.current_congestion) * 0.1
        
        # Speed inversely related to congestion
        self.traffic_system.average_speed = 40 * (1 - self.traffic_system.current_congestion)
        
        # Random incidents
        if np.random.random() < 0.02:  # 2% chance of new incident
            incident = {
                'id': len(self.traffic_system.incidents) + 1,
                'type': np.random.choice(['accident', 'breakdown', 'roadwork']),
                'location': f"Highway {np.random.randint(1, 5)}",
                'severity': np.random.choice(['minor', 'moderate', 'severe']),
                'estimated_clearance': self.current_time + timedelta(hours=np.random.randint(1, 4))
            }
            self.traffic_system.incidents.append(incident)
        
        # Clear resolved incidents
        self.traffic_system.incidents = [
            inc for inc in self.traffic_system.incidents 
            if inc['estimated_clearance'] > self.current_time
        ]
        
        self.traffic_system.road_capacity_utilization = min(1.0, 
            self.traffic_system.current_congestion + len(self.traffic_system.incidents) * 0.1)
    
    def update_kpis(self):
        """Update system KPIs based on current state"""
        # Calculate gate utilization
        operational_gates = sum(1 for s in self.gate_sensors if s.status == 'operational')
        total_processing_capacity = operational_gates * 4  # 4 trucks per hour per gate
        current_load = sum(s.queue_length for s in self.gate_sensors)
        self.kpis.gate_utilization = current_load / max(1, total_processing_capacity)
        
        # Estimate waiting times based on congestion
        base_waiting_time = 15  # minutes
        congestion_factor = 1 + self.traffic_system.current_congestion * 2
        yard_factor = 1 + self.yard_system.congestion_level * 1.5
        self.kpis.average_waiting_time = base_waiting_time * congestion_factor * yard_factor
        
        # Customer satisfaction based on waiting time and reliability
        if self.kpis.average_waiting_time < 15:
            satisfaction_base = 0.95
        elif self.kpis.average_waiting_time < 30:
            satisfaction_base = 0.85
        else:
            satisfaction_base = 0.70
        
        # Adjust for gate malfunctions
        malfunction_penalty = len([s for s in self.gate_sensors if s.status == 'malfunction']) * 0.05
        self.kpis.customer_satisfaction = max(0.5, satisfaction_base - malfunction_penalty)
        
        # System efficiency
        gate_efficiency = operational_gates / self.num_gates
        yard_efficiency = 1 - self.yard_system.congestion_level
        traffic_efficiency = 1 - self.traffic_system.current_congestion
        self.kpis.system_efficiency = (gate_efficiency + yard_efficiency + traffic_efficiency) / 3
        
        # Carbon emissions (simplified calculation)
        idling_factor = self.traffic_system.current_congestion * 50
        base_emissions = 100
        self.kpis.carbon_emissions = base_emissions + idling_factor
    
    def step(self):
        """Advance simulation by one time step"""
        self.current_time += self.time_step
        
        # Update all system components
        new_requests = self.generate_truck_requests()
        for request in new_requests:
            self.truck_requests.append(request)
        
        self.update_gate_sensors()
        self.update_yard_system()
        self.update_traffic_system()
        self.update_kpis()
        
        # Store sensor readings
        reading = {
            'timestamp': self.current_time,
            'gate_status': [s.status for s in self.gate_sensors],
            'yard_utilization': self.yard_system.yard_utilization,
            'traffic_congestion': self.traffic_system.current_congestion,
            'kpis': {
                'waiting_time': self.kpis.average_waiting_time,
                'satisfaction': self.kpis.customer_satisfaction,
                'efficiency': self.kpis.system_efficiency
            }
        }
        self.sensor_readings.append(reading)
        
        return reading

print("Real-time data simulator defined")

Real-time data simulator defined


In [ ]:
# Predictive Analytics Engine
class PredictiveAnalytics:
    """Predictive analytics for demand forecasting and congestion prediction"""
    
    def __init__(self):
        self.historical_patterns = {}
        self.demand_models = {}
        self.congestion_models = {}
        
    def analyze_historical_patterns(self, sensor_data: List[Dict]) -> Dict:
        """Analyze historical patterns from sensor data"""
        if not sensor_data:
            return {}
        
        # Extract time series data
        timestamps = [reading['timestamp'] for reading in sensor_data]
        waiting_times = [reading['kpis']['waiting_time'] for reading in sensor_data]
        congestions = [reading['kpis']['efficiency'] for reading in sensor_data]
        satisfactions = [reading['kpis']['satisfaction'] for reading in sensor_data]
        
        patterns = {
            'avg_waiting_time': np.mean(waiting_times),
            'peak_waiting_time': np.max(waiting_times),
            'waiting_time_trend': self._calculate_trend(waiting_times),
            'avg_congestion': np.mean(congestions),
            'congestion_trend': self._calculate_trend(congestions),
            'avg_satisfaction': np.mean(satisfactions),
            'satisfaction_trend': self._calculate_trend(satisfactions),
            'volatility': np.std(waiting_times)
        }
        
        return patterns
    
    def _calculate_trend(self, values: List[float]) -> float:
        """Calculate simple linear trend"""
        if len(values) < 2:
            return 0.0
        
        x = np.arange(len(values))
        y = np.array(values)
        
        # Simple linear regression
        slope = np.polyfit(x, y, 1)[0]
        return slope
    
    def forecast_demand(self, current_requests: int, historical_data: List[Dict], 
                       forecast_horizon: int = 4) -> Dict:
        """Forecast demand for future time slots"""
        
        # Simple forecasting based on historical patterns and current trends
        if not historical_data:
            # Default forecast if no historical data
            return {
                'forecast': [current_requests] * forecast_horizon,
                'confidence': [0.5] * forecast_horizon,
                'method': 'default'
            }
        
        # Extract recent demand patterns
        recent_demand = [len(reading.get('new_requests', [])) for reading in historical_data[-10:]]
        
        if len(recent_demand) < 2:
            avg_demand = current_requests
            demand_trend = 0
        else:
            avg_demand = np.mean(recent_demand)
            demand_trend = self._calculate_trend(recent_demand)
        
        # Generate forecast
        forecast = []
        confidence = []
        
        for i in range(forecast_horizon):
            # Base forecast with trend adjustment
            predicted = avg_demand + demand_trend * (i + 1)
            
            # Add seasonal adjustment (simplified)
            seasonal_factor = 1.0 + 0.2 * np.sin(2 * np.pi * i / 8)  # Daily pattern
            predicted *= seasonal_factor
            
            # Ensure non-negative
            predicted = max(0, predicted)
            
            # Confidence decreases with horizon
            conf = max(0.3, 0.9 - i * 0.15)
            
            forecast.append(predicted)
            confidence.append(conf)
        
        return {
            'forecast': forecast,
            'confidence': confidence,
            'method': 'trend_based',
            'avg_historical': avg_demand,
            'trend': demand_trend
        }
    
    def predict_congestion(self, current_congestion: float, 
                          historical_data: List[Dict],
                          forecast_horizon: int = 4) -> Dict:
        """Predict congestion for future time slots"""
        
        if not historical_data:
            return {
                'forecast': [current_congestion] * forecast_horizon,
                'risk_level': 'medium',
                'factors': []
            }
        
        # Extract congestion patterns
        recent_congestion = [reading['kpis']['efficiency'] for reading in historical_data[-20:]]
        
        if len(recent_congestion) < 2:
            base_congestion = current_congestion
            congestion_trend = 0
        else:
            base_congestion = np.mean(recent_congestion)
            congestion_trend = self._calculate_trend(recent_congestion)
        
        # Generate congestion forecast
        forecast = []
        
        for i in range(forecast_horizon):
            predicted = base_congestion + congestion_trend * (i + 1)
            
            # Add time-of-day patterns
            hour_factor = 0.1 * np.sin(2 * np.pi * i / 8 - np.pi/2)  # Peak patterns
            predicted += hour_factor
            
            # Clamp to valid range
            predicted = max(0.2, min(1.0, predicted))
            
            forecast.append(predicted)
        
        # Determine risk level
        max_predicted = max(forecast)
        if max_predicted > 0.8:
            risk_level = 'high'
        elif max_predicted > 0.6:
            risk_level = 'medium'
        else:
            risk_level = 'low'
        
        # Identify contributing factors
        factors = []
        if congestion_trend > 0.01:
            factors.append('Increasing trend')
        if base_congestion > 0.7:
            factors.append('High baseline congestion')
        if max(forecast) - min(forecast) > 0.3:
            factors.append('High volatility')
        
        return {
            'forecast': forecast,
            'risk_level': risk_level,
            'factors': factors,
            'trend': congestion_trend
        }
    
    def detect_anomalies(self, current_data: Dict, historical_patterns: Dict) -> List[Dict]:
        """Detect anomalies in current system state"""
        anomalies = []
        
        if not historical_patterns:
            return anomalies
        
        # Check waiting time anomaly
        current_waiting = current_data.get('waiting_time', 0)
        avg_waiting = historical_patterns.get('avg_waiting_time', 0)
        
        if current_waiting > avg_waiting * 1.5:
            anomalies.append({
                'type': 'high_waiting_time',
                'severity': 'high' if current_waiting > avg_waiting * 2 else 'medium',
                'current': current_waiting,
                'expected': avg_waiting,
                'description': f'Waiting time {current_waiting:.1f}min is significantly above average {avg_waiting:.1f}min'
            })
        
        # Check satisfaction anomaly
        current_satisfaction = current_data.get('satisfaction', 1.0)
        avg_satisfaction = historical_patterns.get('avg_satisfaction', 0.9)
        
        if current_satisfaction < avg_satisfaction - 0.1:
            anomalies.append({
                'type': 'low_satisfaction',
                'severity': 'high' if current_satisfaction < avg_satisfaction - 0.2 else 'medium',
                'current': current_satisfaction,
                'expected': avg_satisfaction,
                'description': f'Customer satisfaction {current_satisfaction:.2f} is below average {avg_satisfaction:.2f}'
            })
        
        return anomalies

print("Predictive analytics engine defined")

Predictive analytics engine defined


In [ ]:
try:
    # Digital Twin Core System
    class DigitalTwinSystem:
        """Main Digital Twin system integrating all components"""
        
        def __init__(self, num_gates: int = 8):
            self.simulator = RealTimeDataSimulator(num_gates)
            self.analytics = PredictiveAnalytics()
            
            # System state
           .current_time = datetime.now()
            self.historical_data = []
            self.appointment_schedule = {}
            
            # Optimization parameters
           .optimization_enabled = True
           .auto_reschedule = True
            
            # Performance tracking
           .optimization_history = []
            
        def initialize_system(self, warmup_steps: int = 50):
            """Initialize system with warmup data"""
            print(f"Initializing Digital Twin with {warmup_steps} warmup steps...")
            
            for step in range(warmup_steps):
                reading = self.simulator.step()
                self.historical_data.append(reading)
                
                if step % 10 == 0:
                    print(f"  Warmup step {step}/{warmup_steps} completed")
            
            print("✓ Digital Twin initialization completed")
            return self.historical_data[-1]  # Return latest state
        
        def get_current_state(self) -> Dict:
            """Get comprehensive current system state"""
            
            # Latest sensor reading
            if self.historical_data:
                latest_reading = self.historical_data[-1]
            else:
                latest_reading = self.simulator.step()
                self.historical_data.append(latest_reading)
            
            # Current system components
            state = {
                'timestamp': self.simulator.current_time,
                'gates': {
                    'total': len(self.simulator.gate_sensors),
                    'operational': sum(1 for s in self.simulator.gate_sensors if s.status == 'operational'),
                    'malfunctions': [s.gate_id for s in self.simulator.gate_sensors if s.status == 'malfunction'],
                    'queue_lengths': [s.queue_length for s in self.simulator.gate_sensors],
                    'utilization': [s.utilization_24h for s in self.simulator.gate_sensors]
                },
                'yard': {
                    'utilization': self.simulator.yard_system.yard_utilization,
                    'available_spaces': self.simulator.yard_system.available_spaces,
                    'equipment': self.simulator.yard_system.equipment_availability,
                    'congestion': self.simulator.yard_system.congestion_level,
                    'weather_impact': self.simulator.yard_system.weather_impact
                },
                'traffic': {
                    'congestion': self.simulator.traffic_system.current_congestion,
                    'average_speed': self.simulator.traffic_system.average_speed,
                    'incidents': len(self.simulator.traffic_system.incidents),
                    'weather': self.simulator.traffic_system.weather_condition,
                    'capacity_utilization': self.simulator.traffic_system.road_capacity_utilization
                },
                'kpis': {
                    'waiting_time': self.simulator.kpis.average_waiting_time,
                    'satisfaction': self.simulator.kpis.customer_satisfaction,
                    'efficiency': self.simulator.kpis.system_efficiency,
                    'gate_utilization': self.simulator.kpis.gate_utilization,
                    'carbon_emissions': self.simulator.kpis.carbon_emissions
                },
                'pending_requests': len(self.simulator.truck_requests)
            }
            
            return state
        
        def run_predictive_analysis(self) -> Dict:
            """Run predictive analytics on current data"""
            
            # Analyze historical patterns
            patterns = self.analytics.analyze_historical_patterns(self.historical_data)
            
            # Forecast demand
            current_requests = len(self.simulator.truck_requests)
            demand_forecast = self.analytics.forecast_demand(
                current_requests, self.historical_data, forecast_horizon=4
            )
            
            # Predict congestion
            current_congestion = self.simulator.traffic_system.current_congestion
            congestion_forecast = self.analytics.predict_congestion(
                current_congestion, self.historical_data, forecast_horizon=4
            )
            
            # Detect anomalies
            current_kpis = {
                'waiting_time': self.simulator.kpis.average_waiting_time,
                'satisfaction': self.simulator.kpis.customer_satisfaction,
                'efficiency': self.simulator.kpis.system_efficiency
            }
            anomalies = self.analytics.detect_anomalies(current_kpis, patterns)
            
            return {
                'patterns': patterns,
                'demand_forecast': demand_forecast,
                'congestion_forecast': congestion_forecast,
                'anomalies': anomalies,
                'analysis_timestamp': self.simulator.current_time
            }
        
        def optimize_appointments(self, scenario: str = 'baseline') -> Dict:
            """Run appointment optimization based on current state and predictions"""
            
            print(f"\nRunning appointment optimization ({scenario} scenario)...")
            
            # Get current state and predictions
            current_state = self.get_current_state()
            predictions = self.run_predictive_analysis()
            
            # Extract pending requests
            pending_requests = list(self.simulator.truck_requests)
            
            if not pending_requests:
                return {'status': 'no_requests', 'actions': []}
            
            # Optimization logic based on scenario
            actions = []
            
            if scenario == 'baseline':
                actions = self._baseline_optimization(pending_requests, current_state, predictions)
            elif scenario == 'congestion_mitigation':
                actions = self._congestion_optimization(pending_requests, current_state, predictions)
            elif scenario == 'equipment_failure':
                actions = self._equipment_optimization(pending_requests, current_state, predictions)
            elif scenario == 'weather_adaptation':
                actions = self._weather_optimization(pending_requests, current_state, predictions)
            
            # Record optimization
            optimization_record = {
                'timestamp': self.simulator.current_time,
                'scenario': scenario,
                'state': current_state,
                'predictions': predictions,
                'actions': actions,
                'impact': self._calculate_optimization_impact(actions)
            }
            
            self.optimization_history.append(optimization_record)
            
            return optimization_record
        
        def _baseline_optimization(self, requests: List[TruckRequest], 
                                  state: Dict, predictions: Dict) -> List[Dict]:
            """Baseline optimization using standard rules"""
            actions = []
            
            # Sort requests by priority
            sorted_requests = sorted(requests, key=lambda r: r.weight, reverse=True)
            
            # Simple assignment based on preferred times and capacity
            gate_capacity = state['gates']['operational'] * 4  # 4 trucks per hour per gate
            
            for request in sorted_requests:
                preferred_slot = request.preferred_time % 8  # 8 time slots
                
                # Check if preferred slot has capacity
                current_assignments = sum(1 for r in requests 
                                         if hasattr(r, 'assigned_slot') and r.assigned_slot == preferred_slot)
                
                if current_assignments < gate_capacity:
                    request.assigned_slot = preferred_slot
                    actions.append({
                        'type': 'assignment',
                        'request_id': request.id,
                        'assigned_slot': preferred_slot,
                        'method': 'preferred_time',
                        'reason': 'Capacity available in preferred slot'
                    })
                else:
                    # Find alternative slot
                    for slot in range(8):
                        slot_assignments = sum(1 for r in requests 
                                               if hasattr(r, 'assigned_slot') and r.assigned_slot == slot)
                        if slot_assignments < gate_capacity:
                            request.assigned_slot = slot
                            actions.append({
                                'type': 'assignment',
                                'request_id': request.id,
                                'assigned_slot': slot,
                                'method': 'alternative_slot',
                                'reason': f'Preferred slot full, assigned to slot {slot}'
                            })
                            break
            
            return actions
        
        def _congestion_optimization(self, requests: List[TruckRequest], 
                                    state: Dict, predictions: Dict) -> List[Dict]:
            """Optimization focused on congestion mitigation"""
            actions = []
            
            congestion_forecast = predictions['congestion_forecast']['forecast']
            
            # Identify high-congestion periods
            high_congestion_slots = [i for i, cong in enumerate(congestion_forecast) if cong > 0.7]
            
            # Redistribute requests away from high congestion periods
            for request in requests:
                preferred_slot = request.preferred_time % 8
                
                if preferred_slot in high_congestion_slots:
                    # Find low-congestion alternative
                    low_congestion_slots = [i for i, cong in enumerate(congestion_forecast) if cong < 0.5]
                    
                    if low_congestion_slots:
                        alternative_slot = min(low_congestion_slots, key=lambda s: abs(s - preferred_slot))
                        request.assigned_slot = alternative_slot
                        
                        actions.append({
                            'type': 'congestion_redistribution',
                            'request_id': request.id,
                            'original_slot': preferred_slot,
                            'assigned_slot': alternative_slot,
                            'congestion_reduction': congestion_forecast[preferred_slot] - congestion_forecast[alternative_slot],
                            'reason': 'Redistributed from high to low congestion period'
                        })
                    else:
                        # No good alternative, keep original
                        request.assigned_slot = preferred_slot
                else:
                    request.assigned_slot = preferred_slot
            
            return actions
        
        def _equipment_optimization(self, requests: List[TruckRequest], 
                                    state: Dict, predictions: Dict) -> List[Dict]:
            """Optimization for equipment failure scenarios"""
            actions = []
            
            malfunction_gates = state['gates']['malfunctions']
            operational_gates = state['gates']['operational']
            
            if malfunction_gates:
                # Reduce effective capacity
                effective_capacity = operational_gates * 3  # Reduced capacity due to strain
                
                actions.append({
                    'type': 'capacity_adjustment',
                    'malfunction_gates': malfunction_gates,
                    'operational_gates': operational_gates,
                    'effective_capacity': effective_capacity,
                    'reason': f'Capacity reduced due to {len(malfunction_gates)} gate malfunctions'
                })
                
                # Prioritize high-weight requests
                sorted_requests = sorted(requests, key=lambda r: r.weight, reverse=True)
                
                assigned_count = 0
                for request in sorted_requests:
                    if assigned_count < effective_capacity:
                        request.assigned_slot = request.preferred_time % 8
                        assigned_count += 1
                    else:
                        # Delay lower priority requests
                        actions.append({
                            'type': 'delay',
                            'request_id': request.id,
                            'weight': request.weight,
                            'reason': 'Delayed due to reduced capacity from equipment failure'
                        })
            else:
                # Normal operation
                for request in requests:
                    request.assigned_slot = request.preferred_time % 8
            
            return actions
        
        def _weather_optimization(self, requests: List[TruckRequest], 
                                  state: Dict, predictions: Dict) -> List[Dict]:
            """Optimization for weather-related impacts"""
            actions = []
            
            weather_impact = state['yard']['weather_impact']
            weather_condition = state['traffic']['weather']
            
            if weather_impact < 1.0:  # Adverse weather
                # Reduce processing rates
                processing_rate = weather_impact
                
                actions.append({
                    'type': 'weather_adaptation',
                    'condition': weather_condition,
                    'impact_factor': weather_impact,
                    'processing_rate': processing_rate,
                    'reason': f'Processing rate adjusted to {processing_rate:.1%} due to {weather_condition}'
                })
                
                # Spread out appointments to reduce congestion
                for request in requests:
                    preferred_slot = request.preferred_time % 8
                    
                    # Add buffer time between appointments
                    buffer_slots = int(2 / processing_rate)  # More buffer for worse weather
                    adjusted_slot = (preferred_slot + buffer_slots) % 8
                    
                    request.assigned_slot = adjusted_slot
                    
                    if adjusted_slot != preferred_slot:
                        actions.append({
                            'type': 'weather_redistribution',
                            'request_id': request.id,
                            'original_slot': preferred_slot,
                            'assigned_slot': adjusted_slot,
                            'reason': f'Adjusted for {weather_condition} conditions'
                        })
            else:
                # Normal weather
                for request in requests:
                    request.assigned_slot = request.preferred_time % 8
            
            return actions
        
        def _calculate_optimization_impact(self, actions: List[Dict]) -> Dict:
            """Calculate the impact of optimization actions"""
            
            impact = {
                'total_actions': len(actions),
                'redistributed_requests': 0,
                'delayed_requests': 0,
                'congestion_reduction': 0.0,
                'capacity_adjustment': 0,
                'weather_adaptations': 0
            }
            
            for action in actions:
                if action['type'] in ['congestion_redistribution', 'weather_redistribution']:
                    impact['redistributed_requests'] += 1
                    if 'congestion_reduction' in action:
                        impact['congestion_reduction'] += action['congestion_reduction']
                elif action['type'] == 'delay':
                    impact['delayed_requests'] += 1
                elif action['type'] == 'capacity_adjustment':
                    impact['capacity_adjustment'] = action['effective_capacity']
                elif action['type'] == 'weather_adaptation':
                    impact['weather_adaptations'] += 1
            
            return impact
        
        def run_what_if_scenario(self, scenario_name: str, 
                               scenario_params: Dict) -> Dict:
            """Run what-if scenario analysis"""
            
            print(f"\nRunning what-if scenario: {scenario_name}")
            print(f"Parameters: {scenario_params}")
            
            # Save current state
            original_state = {
                'gates': self.simulator.gate_sensors.copy(),
                'yard': self.simulator.yard_system,
                'traffic': self.simulator.traffic_system,
                'kpis': self.simulator.kpis
            }
            
            # Apply scenario modifications
            modifications = []
            
            if scenario_name == 'gate_failure':
                failed_gate = scenario_params.get('gate_id', 0)
                self.simulator.gate_sensors[failed_gate].status = 'malfunction'
                modifications.append(f"Gate {failed_gate} set to malfunction")
            
            elif scenario_name == 'traffic_incident':
                severity = scenario_params.get('severity', 'moderate')
                congestion_increase = {'minor': 0.2, 'moderate': 0.4, 'severe': 0.6}[severity]
                self.simulator.traffic_system.current_congestion = min(1.0, 
                    self.simulator.traffic_system.current_congestion + congestion_increase)
                modifications.append(f"Traffic congestion increased by {congestion_increase:.1%}")
            
            elif scenario_name == 'weather_event':
                weather_type = scenario_params.get('weather', 'rain')
                self.simulator.traffic_system.weather_condition = weather_type
                modifications.append(f"Weather changed to {weather_type}")
            
            elif scenario_name == 'demand_surge':
                surge_factor = scenario_params.get('surge_factor', 1.5)
                current_requests = len(self.simulator.truck_requests)
                additional_requests = int(current_requests * (surge_factor - 1))
                
                new_requests = self.simulator.generate_truck_requests(additional_requests)
                for req in new_requests:
                    self.simulator.truck_requests.append(req)
                
                modifications.append(f"Added {additional_requests} additional requests ({surge_factor:.1f}x surge)")
            
            # Run optimization with scenario
            optimization_result = self.optimize_appointments(scenario=f'what_if_{scenario_name}')
            
            # Calculate scenario impact
            scenario_impact = {
                'scenario_name': scenario_name,
                'modifications': modifications,
                'optimization_actions': optimization_result.get('actions', []),
                'impact_metrics': optimization_result.get('impact', {}),
                'resulting_kpis': self.get_current_state()['kpis']
            }
            
            # Restore original state (simplified)
            self.simulator.gate_sensors = original_state['gates']
            self.simulator.yard_system = original_state['yard']
            self.simulator.traffic_system = original_state['traffic']
            self.simulator.kpis = original_state['kpis']
            
            return scenario_impact
    
    print("Digital Twin core system defined")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: unindent does not match any outer indentation level (<string>, line 10)...]

In [ ]:
try:
    # Demonstrate Digital Twin capabilities
    def demonstrate_digital_twin():
        """Demonstrate comprehensive Digital Twin capabilities"""
        
        print("=" * 80)
        print("DIGITAL TWIN SYSTEM DEMONSTRATION")
        print("=" * 80)
        
        # Initialize Digital Twin
        digital_twin = DigitalTwinSystem(num_gates=8)
        
        print("\n1. System Initialization:")
        initial_state = digital_twin.initialize_system(warmup_steps=20)
        
        print(f"   Current time: {digital_twin.simulator.current_time.strftime('%H:%M')}")
        print(f"   Historical data points: {len(digital_twin.historical_data)}")
        print(f"   Pending requests: {len(digital_twin.simulator.truck_requests)}")
        
        # Display current system state
        print("\n2. Current System State:")
        current_state = digital_twin.get_current_state()
        
        print(f"   Gates: {current_state['gates']['operational']}/{current_state['gates']['total']} operational")
        if current_state['gates']['malfunctions']:
            print(f"   Malfunctions: Gates {current_state['gates']['malfunctions']}")
        print(f"   Yard utilization: {current_state['yard']['utilization']:.1%}")
        print(f"   Traffic congestion: {current_state['traffic']['congestion']:.1%}")
        print(f"   Weather: {current_state['traffic']['weather']}")
        print(f"   Average waiting time: {current_state['kpis']['waiting_time']:.1f} minutes")
        print(f"   Customer satisfaction: {current_state['kpis']['satisfaction']:.1%}")
        print(f"   System efficiency: {current_state['kpis']['efficiency']:.1%}")
        
        # Run predictive analytics
        print("\n3. Predictive Analytics:")
        predictions = digital_twin.run_predictive_analysis()
        
        print(f"   Demand forecast (next 4 slots): {predictions['demand_forecast']['forecast']}")
        print(f"   Congestion risk level: {predictions['congestion_forecast']['risk_level']}")
        print(f"   Congestion forecast: {[f'{c:.2f}' for c in predictions['congestion_forecast']['forecast'][:4]]}")
        
        if predictions['anomalies']:
            print(f"   Anomalies detected: {len(predictions['anomalies'])}")
            for anomaly in predictions['anomalies'][:2]:
                print(f"     - {anomaly['type']}: {anomaly['description']}")
        else:
            print("   No anomalies detected")
        
        # Run baseline optimization
        print("\n4. Baseline Optimization:")
        baseline_optimization = digital_twin.optimize_appointments(scenario='baseline')
        
        baseline_actions = baseline_optimization.get('actions', [])
        baseline_impact = baseline_optimization.get('impact', {})
        
        print(f"   Total actions: {baseline_impact.get('total_actions', 0)}")
        print(f"   Redistributed requests: {baseline_impact.get('redistributed_requests', 0)}")
        print(f"   Delayed requests: {baseline_impact.get('delayed_requests', 0)}")
        
        # Show sample actions
        if baseline_actions:
            print("   Sample actions:")
            for action in baseline_actions[:3]:
                if action['type'] == 'assignment':
                    print(f"     - Request {action['request_id']} assigned to slot {action['assigned_slot']}")
                else:
                    print(f"     - {action['type']}: {action.get('reason', 'N/A')}")
        
        return digital_twin, current_state, predictions
    
    digital_twin, current_state, predictions = demonstrate_digital_twin()
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'DigitalTwinSystem' is not defined...]

In [ ]:
try:
    # What-if scenario analysis
    def run_what_if_scenarios(digital_twin):
        """Run comprehensive what-if scenario analysis"""
        
        print("\n" + "=" * 80)
        print("WHAT-IF SCENARIO ANALYSIS")
        print("=" * 80)
        
        scenarios = [
            {
                'name': 'Peak Morning Rush',
                'type': 'demand_surge',
                'params': {'surge_factor': 2.0},
                'description': 'Double the normal request volume during peak hours'
            },
            {
                'name': 'Gate Malfunction',
                'type': 'gate_failure',
                'params': {'gate_id': 3},
                'description': 'Critical gate 3 experiences malfunction'
            },
            {
                'name': 'Traffic Incident',
                'type': 'traffic_incident',
                'params': {'severity': 'severe'},
                'description': 'Major traffic incident increases congestion'
            },
            {
                'name': 'Adverse Weather',
                'type': 'weather_event',
                'params': {'weather': 'rain'},
                'description': 'Heavy rain reduces processing efficiency'
            }
        ]
        
        scenario_results = {}
        
        for scenario in scenarios:
            print(f"\n--- Scenario: {scenario['name']} ---")
            print(f"Description: {scenario['description']}")
            
            # Run scenario
            result = digital_twin.run_what_if_scenario(
                scenario['type'], scenario['params']
            )
            
            scenario_results[scenario['name']] = result
            
            # Display results
            print(f"\nModifications applied:")
            for mod in result['modifications']:
                print(f"  - {mod}")
            
            impact = result['impact_metrics']
            print(f"\nOptimization impact:")
            print(f"  - Total actions: {impact.get('total_actions', 0)}")
            print(f"  - Redistributed requests: {impact.get('redistributed_requests', 0)}")
            print(f"  - Delayed requests: {impact.get('delayed_requests', 0)}")
            
            kpis = result['resulting_kpis']
            print(f"\nResulting KPIs:")
            print(f"  - Waiting time: {kpis['waiting_time']:.1f} minutes")
            print(f"  - Customer satisfaction: {kpis['satisfaction']:.1%}")
            print(f"  - System efficiency: {kpis['efficiency']:.1%}")
            print(f"  - Carbon emissions: {kpis['carbon_emissions']:.1f} kg CO2")
        
        # Scenario comparison
        print("\n" + "="*80)
        print("SCENARIO COMPARISON SUMMARY")
        print("="*80)
        
        print("\nScenario                | Waiting Time | Satisfaction | Efficiency | Actions")
        print("-" * 75)
        
        baseline_kpis = current_state['kpis']
        print(f"{'Baseline':>22} | {baseline_kpis['waiting_time']:>11.1f} | {baseline_kpis['satisfaction']:>11.1%} | {baseline_kpis['efficiency']:>9.1%} | {'N/A':>7}")
        
        for scenario_name, result in scenario_results.items():
            kpis = result['resulting_kpis']
            actions = result['impact_metrics'].get('total_actions', 0)
            
            # Calculate improvements
            waiting_improvement = baseline_kpis['waiting_time'] - kpis['waiting_time']
            waiting_change = f"{waiting_improvement:+.1f}" if waiting_improvement != 0 else "0.0"
            
            print(f"{scenario_name:>22} | {kpis['waiting_time']:>11.1f} ({waiting_change:>5}) | {kpis['satisfaction']:>11.1%} | {kpis['efficiency']:>9.1%} | {actions:>7}")
        
        return scenario_results
    
    scenario_results = run_what_if_scenarios(digital_twin)
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'digital_twin' is not defined...]

In [ ]:
try:
    # Comprehensive visualization of Digital Twin performance
    def visualize_digital_twin_performance(digital_twin, scenario_results):
        """Create comprehensive visualizations of Digital Twin performance"""
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        fig.suptitle('Digital Twin System Performance Analysis', fontsize=16, fontweight='bold')
        
        # 1. Real-time KPI monitoring
        ax1 = axes[0, 0]
        if len(digital_twin.historical_data) > 1:
            timestamps = [reading['timestamp'] for reading in digital_twin.historical_data[-20:]]
            waiting_times = [reading['kpis']['waiting_time'] for reading in digital_twin.historical_data[-20:]]
            satisfactions = [reading['kpis']['satisfaction'] * 100 for reading in digital_twin.historical_data[-20:]]
            efficiencies = [reading['kpis']['efficiency'] * 100 for reading in digital_twin.historical_data[-20:]]
            
            ax1.plot(timestamps, waiting_times, 'r-', label='Waiting Time (min)', linewidth=2)
            ax1_twin = ax1.twinx()
            ax1_twin.plot(timestamps, satisfactions, 'g-', label='Satisfaction (%)', linewidth=2)
            ax1_twin.plot(timestamps, efficiencies, 'b-', label='Efficiency (%)', linewidth=2)
            
            ax1.set_xlabel('Time')
            ax1.set_ylabel('Waiting Time (min)', color='r')
            ax1_twin.set_ylabel('Percentage (%)', color='g')
            ax1.set_title('Real-time KPI Monitoring', fontweight='bold')
            ax1.legend(loc='upper left')
            ax1_twin.legend(loc='upper right')
            ax1.grid(True, alpha=0.3)
            
            # Format x-axis
            import matplotlib.dates as mdates
            ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
            fig.autofmt_xdate()
        
        # 2. Predictive Analytics Visualization
        ax2 = axes[0, 1]
        demand_forecast = predictions['demand_forecast']
        congestion_forecast = predictions['congestion_forecast']
        
        slots = range(len(demand_forecast['forecast']))
        
        ax2.bar([s - 0.2 for s in slots], demand_forecast['forecast'], 0.4, 
               label='Demand Forecast', alpha=0.7, color='blue')
        ax2.bar([s + 0.2 for s in slots], [c * 10 for c in congestion_forecast['forecast']], 0.4,
               label='Congestion Index (x10)', alpha=0.7, color='red')
        
        ax2.set_xlabel('Future Time Slots')
        ax2.set_ylabel('Demand / Congestion')
        ax2.set_title('Predictive Analytics Forecast', fontweight='bold')
        ax2.set_xticks(slots)
        ax2.set_xticklabels([f'T+{i}' for i in slots])
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # 3. System Component Status
        ax3 = axes[0, 2]
        state = digital_twin.get_current_state()
        
        # Gate status
        gate_statuses = ['Operational', 'Malfunction']
        gate_counts = [state['gates']['operational'], len(state['gates']['malfunctions'])]
        colors_gate = ['green', 'red']
        
        wedges, texts, autotexts = ax3.pie(gate_counts, labels=gate_statuses, colors=colors_gate,
                                             autopct='%1.0f', startangle=90)
        ax3.set_title('Gate Status Distribution', fontweight='bold')
        
        # 4. Scenario Impact Comparison
        ax4 = axes[1, 0]
        scenarios = list(scenario_results.keys())
        baseline_waiting = [current_state['kpis']['waiting_time']] * len(scenarios)
        scenario_waiting = [result['resulting_kpis']['waiting_time'] for result in scenario_results.values()]
        
        x = np.arange(len(scenarios))
        width = 0.35
        
        ax4.bar(x - width/2, baseline_waiting, width, label='Baseline', alpha=0.7, color='lightblue')
        ax4.bar(x + width/2, scenario_waiting, width, label='Scenario', alpha=0.7, color='orange')
        
        ax4.set_xlabel('Scenarios')
        ax4.set_ylabel('Waiting Time (minutes)')
        ax4.set_title('Scenario Impact on Waiting Time', fontweight='bold')
        ax4.set_xticks(x)
        ax4.set_xticklabels([s[:8] for s in scenarios], rotation=45)
        ax4.legend()
        ax4.grid(True, alpha=0.3)
        
        # 5. Optimization Actions Breakdown
        ax5 = axes[1, 1]
        action_types = ['Redistributed', 'Delayed', 'Capacity Adj.', 'Weather Adapt.']
        action_counts = [0, 0, 0, 0]
        
        for result in scenario_results.values():
            impact = result['impact_metrics']
            action_counts[0] += impact.get('redistributed_requests', 0)
            action_counts[1] += impact.get('delayed_requests', 0)
            action_counts[2] += 1 if impact.get('capacity_adjustment', 0) > 0 else 0
            action_counts[3] += impact.get('weather_adaptations', 0)
        
        colors_actions = ['skyblue', 'lightcoral', 'lightgreen', 'gold']
        bars = ax5.bar(action_types, action_counts, color=colors_actions, alpha=0.8)
        
        ax5.set_xlabel('Action Types')
        ax5.set_ylabel('Total Count')
        ax5.set_title('Optimization Actions Breakdown', fontweight='bold')
        ax5.grid(True, alpha=0.3)
        
        # Add value labels on bars
        for bar, count in zip(bars, action_counts):
            if count > 0:
                ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                        str(count), ha='center', va='bottom', fontweight='bold')
        
        # 6. System Efficiency Dashboard
        ax6 = axes[1, 2]
        # Create a comprehensive efficiency gauge
        efficiency_metrics = {
            'Gate Efficiency': state['gates']['operational'] / state['gates']['total'] * 100,
            'Yard Efficiency': (1 - state['yard']['congestion']) * 100,
            'Traffic Efficiency': (1 - state['traffic']['congestion']) * 100,
            'Overall Efficiency': state['kpis']['efficiency'] * 100
        }
        
        metrics = list(efficiency_metrics.keys())
        values = list(efficiency_metrics.values())
        
        colors = ['green' if v > 80 else 'orange' if v > 60 else 'red' for v in values]
        bars = ax6.barh(metrics, values, color=colors, alpha=0.8)
        
        ax6.set_xlim(0, 100)
        ax6.set_xlabel('Efficiency (%)')
        ax6.set_title('System Efficiency Dashboard', fontweight='bold')
        ax6.grid(True, alpha=0.3)
        
        # Add value labels
        for bar, value in zip(bars, values):
            ax6.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                    f'{value:.1f}%', ha='left', va='center', fontweight='bold')
        
        plt.tight_layout()
        plt.show()
        
        # Print detailed analysis
        print("\n" + "=" * 80)
        print("DIGITAL TWIN PERFORMANCE ANALYSIS")
        print("=" * 80)
        
        print("\n1. System Health Assessment:")
        health_score = (state['gates']['operational'] / state['gates']['total'] + 
                       (1 - state['yard']['congestion']) + 
                       (1 - state['traffic']['congestion'])) / 3 * 100
        
        print(f"   Overall Health Score: {health_score:.1f}%")
        if health_score > 80:
            print("   ✓ System operating in excellent condition")
        elif health_score > 60:
            print("   ⚠ System operating with some concerns")
        else:
            print("   ✗ System requires immediate attention")
        
        print("\n2. Predictive Capability Assessment:")
        demand_volatility = np.std(demand_forecast['forecast']) if len(demand_forecast['forecast']) > 1 else 0
        congestion_risk = congestion_forecast['risk_level']
        
        print(f"   Demand forecast volatility: {demand_volatility:.2f}")
        print(f"   Congestion risk level: {congestion_risk}")
        
        if len(predictions['anomalies']) > 0:
            print(f"   Active anomalies: {len(predictions['anomalies'])}")
            print("   Recommended immediate action required")
        else:
            print("   ✓ No active anomalies detected")
        
        print("\n3. Optimization Effectiveness:")
        total_optimizations = len(digital_twin.optimization_history)
        if total_optimizations > 0:
            avg_actions = np.mean([opt['impact']['total_actions'] for opt in digital_twin.optimization_history])
            total_redistributions = sum(opt['impact']['redistributed_requests'] for opt in digital_twin.optimization_history)
            
            print(f"   Total optimizations performed: {total_optimizations}")
            print(f"   Average actions per optimization: {avg_actions:.1f}")
            print(f"   Total requests redistributed: {total_redistributions}")
            print("   ✓ Optimization system is actively improving system performance")
        else:
            print("   No optimizations performed yet")
        
        print("\n4. Digital Twin Value Proposition:")
        baseline_emissions = current_state['kpis']['carbon_emissions']
        
        # Calculate potential savings from scenario analysis
        best_scenario = min(scenario_results.keys(), 
                           key=lambda s: scenario_results[s]['resulting_kpis']['carbon_emissions'])
        best_emissions = scenario_results[best_scenario]['resulting_kpis']['carbon_emissions']
        
        emission_reduction = baseline_emissions - best_emissions
        reduction_percentage = emission_reduction / baseline_emissions * 100
        
        print(f"   Baseline carbon emissions: {baseline_emissions:.1f} kg CO2")
        print(f"   Best scenario emissions: {best_emissions:.1f} kg CO2")
        print(f"   Potential reduction: {emission_reduction:.1f} kg CO2 ({reduction_percentage:.1f}%)")
        
        print(f"\n   Best performing scenario: {best_scenario}")
        print("   ✓ Digital Twin enables data-driven decision making")
        print("   ✓ Real-time monitoring prevents performance degradation")
        print("   ✓ Predictive analytics enable proactive optimization")
        print("   ✓ What-if analysis supports strategic planning")
    
    visualize_digital_twin_performance(digital_twin, scenario_results)
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'digital_twin' is not defined...]

### Why this Tier exists vs earlier Tiers
This Tier 5 provides system-of-systems integration capabilities that transcend traditional optimization approaches:

- **Real-time synchronization**: Continuous bi-directional data flow between physical and digital systems
- **Predictive analytics**: Forecast future conditions and enable proactive decision making
- **Multi-system coordination**: Integrates gates, yard, traffic, and external factors holistically
- **What-if analysis**: Test scenarios and strategies without impacting real operations
- **Continuous optimization**: Real-time adjustments based on live system state

### Pros / Cons vs earlier Tiers
**Pros vs Tier 1 (MIP):**
- Handles dynamic, real-time environments
- Incorporates live data and external factors
- Provides continuous monitoring and adaptation
- Supports strategic scenario planning

**Pros vs Tier 2 (EDF):**
- Predictive rather than reactive approach
- Considers multiple interacting systems
- Handles uncertainty and disruptions gracefully
- Provides comprehensive situational awareness

**Pros vs Tier 3 (ACO):**
- Real-time adaptation vs offline optimization
- System-wide coordination vs single objective
- Continuous learning vs static search
- Handles disruptions and突发事件

**Pros vs Tier 4 (ML):**
- Integrates multiple AI techniques holistically
- Real-time execution vs offline training
- System-of-systems vs single model
- Prescriptive vs predictive analytics

**Cons:**
- High implementation complexity and cost
- Requires extensive sensor and data infrastructure
- Significant computational resources for real-time simulation
- Integration challenges with legacy systems
- Maintenance and calibration overhead
- Dependence on data quality and availability

### When to use this Tier
- **Large-scale terminal operations** with complex interactions
- **High-value operations** where optimization benefits justify investment
- **Dynamic environments** requiring real-time adaptation
- **Strategic planning** requiring scenario analysis capabilities
- **Digital transformation initiatives** seeking comprehensive system integration
- **Sustainability-focused operations** requiring environmental impact monitoring
- **Customer-critical services** demanding high reliability and performance